# Step 8: Estimate Kinematic Wave parameters

Create **alpha**, **beta**, and **alpha0** for the Ghana example.
Run from `Example/Example_notebooks` with the **EF5 Builder** kernel.
The root `environment.yml` includes the required model dependencies.

First complete EF5 `BASIN_AVG` in README Step 6 and align Manning roughness
in notebook 4. [Methods and model limitations](../../docs/kinematic_wave.md).


## 1. Imports and paths

Check the paths to the basin-average outputs. Use temperature in °C,
precipitation in mm/year for this example, and basin area in km².
Use the averaged temperature and precipitation files from EF5.


In [1]:
from pathlib import Path
import numpy as np
import rasterio
import matplotlib.pyplot as plt
from rasterio.plot import plotting_extent
from ef5_builder import get_kw_parameters, get_kw_alpha0_from_dem

basic_dir = Path("../data_model/basic")
basin_dir = Path("../posproc_outputs/basin_integration")
output_dir = Path("../data_model/kw_parameters")

dem = basic_dir / "dem.tif"
fac = basic_dir / "fam.tif"
manning = Path("../posproc_outputs/manning/manning_n_2001_1km.tif")

# Expected BASIN_AVG outputs: update filenames if your EF5 run uses other names.
area = basin_dir / "basin.area.tif"
temperature = basin_dir / "mean_temp.tif.avg.tif"
precipitation = basin_dir / "mean_precip.tif.avg.tif"
relief_ratio = basin_dir / "relief.ratio.tif"


## 2. Settings

Keep `precipitation_units="mm/year"` for the Ghana example. `self_fam=True`
means flow accumulation includes the current cell.
With `layer_th=2`, choose a control-file `th` greater than 2.
Set `overwrite=True` to replace existing parameter grids.


In [2]:
precipitation_units = "mm/year"
self_fam = True
layer_th = 2
min_slope = None
overwrite = True


## 3. Check the inputs

Run this cell to check that all files use the same grid as the DEM.
Resolve any reported differences before continuing.


In [3]:
inputs = [dem, fac, area, temperature, precipitation, relief_ratio, manning]
for path in inputs:
    if not path.is_file():
        raise FileNotFoundError(path)

with rasterio.open(dem) as reference:
    grid = (reference.crs, reference.transform, reference.shape)
for path in inputs:
    with rasterio.open(path) as src:
        if src.count != 1 or (src.crs, src.transform, src.shape) != grid:
            raise ValueError(f"Input does not match the DEM grid: {path}")
print("All input grids match the DEM.")


All input grids match the DEM.


## 4. Calculate alpha and beta

Run the model predictions. Only cells with valid inputs and `FAC > layer_th`
receive values. The TH message reminds you of the control-file setting.


In [4]:
kw = get_kw_parameters(
    basin_area=area,
    temperature=temperature,
    precipitation=precipitation,
    relief_ratio=relief_ratio,
    flow_accumulation=fac,
    output_dir=output_dir,
    precipitation_units=precipitation_units,
    self_fam=self_fam,
    layer_th=layer_th,
    overwrite=overwrite,
)
print("Alpha:", kw["alpha"])
print("Beta:", kw["beta"])
print(f"Control-file TH must be greater than {kw['layer_th']}.")


/tmp/ipykernel_19321/1431672351.py:1: UserWarning: Selected layer TH is 2. TH in the EF5 control-file KW parameter block must be strictly greater than this value.
  kw = get_kw_parameters(


Alpha: ../data_model/kw_parameters/kw_alpha.tif
Beta: ../data_model/kw_parameters/kw_beta.tif
Control-file TH must be greater than 2.


## 5. Calculate alpha0

Use Manning roughness and DEM slope to estimate alpha0.
Flat cells remain NoData with `min_slope=None`.
A Manning file named `manning_n_2001_1km.tif` produces `kw_alpha0_2001.tif`;
a file without a year produces `kw_alpha0.tif`.
[Equation and slope method](../../docs/kinematic_wave.md#alpha0-derived-from-the-dem-recommended).


In [6]:
alpha0 = get_kw_alpha0_from_dem(
    manning=manning,
    dem=dem,
    output_dir=output_dir,
    min_slope=min_slope,
    overwrite=overwrite,
)
print("Alpha0:", alpha0)


Alpha0: ../data_model/kw_parameters/kw_alpha0_2001.tif


## 6. Inspect the maps

Check the printed paths, valid-cell counts, and value ranges.
Gray areas are NoData. Alpha and beta cover fewer cells because of the
flow-accumulation threshold and input masks.


In [ ]:
outputs = {"Alpha": kw["alpha"], "Beta": kw["beta"], "Alpha0": alpha0}
fig, axes = plt.subplots(1, 3, figsize=(18, 7), dpi=160)
cmap = plt.get_cmap("viridis").copy()
cmap.set_bad("#e6e6e6")

for ax, (name, path) in zip(axes, outputs.items()):
    with rasterio.open(path) as src:
        assert src.count == 1 and src.dtypes == ("float32",)
        assert src.nodata == -9999
        assert not src.profile.get("tiled", False)
        assert src.compression == rasterio.enums.Compression.deflate
        assert (src.crs, src.transform, src.shape) == grid
        values = np.ma.masked_invalid(src.read(1, masked=True))
        extent = plotting_extent(src)
    count = int(values.count())
    print(f"{name}: {Path(path).resolve()}")
    print(f"  {count:,} valid cells ({count / values.size:.1%})")
    ax.set(title=f"{name} ({count:,} valid cells)", xlabel="Longitude", ylabel="Latitude")
    if count == 0:
        ax.set_facecolor("#e6e6e6")
        ax.text(0.5, 0.5, "No valid predictions", ha="center", va="center", transform=ax.transAxes)
        print("  Check input masks, predictor values, and FAC threshold.")
        continue
    print(f"  Range: {values.min():.6g} to {values.max():.6g}")
    image = ax.imshow(values, extent=extent, origin="upper", cmap=cmap,
                      interpolation="nearest")
    fig.colorbar(image, ax=ax, shrink=0.7)

fig.tight_layout()
plt.show()


## 7. Add the parameters to EF5

Copy the grid paths into your KW parameter block.
Choose `th` greater than `layer_th` and keep `SELFFAM` consistent with the basic grids.


In [7]:
print(f"alpha_grid={kw['alpha']}")
print(f"beta_grid={kw['beta']}")
print(f"alpha0_grid={alpha0}")
print(f"Choose control-file TH > {layer_th}; SELFFAM={str(self_fam).lower()}.")


alpha_grid=../data_model/kw_parameters/kw_alpha.tif
beta_grid=../data_model/kw_parameters/kw_beta.tif
alpha0_grid=../data_model/kw_parameters/kw_alpha0_2001.tif
Choose control-file TH > 2; SELFFAM=true.
